# Exercise: Implementing a Simplified Request Manager

Welcome! In this exercise, you'll build a simplified version of a request manager, inspired by the `LLMEngine` in vLLM. This manager will handle the lifecycle of an inference request, from the moment a user submits a prompt to when the result is ready.

In this exercise you will implement:
*   A method to add new inference requests to a queue.
*   A method to simulate processing a batch of requests.
*   A method to retrieve the output of a completed request.

Let's get started!

In [ ]:
# SETUP CELL: DO NOT MODIFY

import collections
import itertools
from dataclasses import dataclass
from typing import Deque, Dict, Optional

# A simple data class to hold information about a single request.
@dataclass
class Request:
    """Represents a single inference request."""
    request_id: str
    prompt: str
    status: str  # Can be 'pending' or 'completed'

class RequestManager:
    """
    Manages the lifecycle of LLM inference requests.
    
    This class is a simplified simulation of how a service like vLLM's LLMEngine
    might queue and process incoming generation requests.
    """
    def __init__(self):
        """Initializes the RequestManager."""
        # A queue for requests that are waiting to be processed.
        self._request_queue: Deque[Request] = collections.deque()
        
        # A counter to generate unique, sequential request IDs.
        self._request_id_counter = itertools.count()
        
        # A dictionary to store requests that have been processed.
        # Key: request_id (str), Value: Request object
        self._completed_requests: Dict[str, Request] = {}

    def get_request_status(self, request_id: str) -> Optional[str]:
        """
        A helper method to check the status of a request.
        You can use this for debugging.
        """
        # Check if the request is in the completed dictionary
        if request_id in self._completed_requests:
            return self._completed_requests[request_id].status

        # If not, check if it's still pending in the queue
        for req in self._request_queue:
            if req.request_id == request_id:
                return req.status
        
        # If the request_id is not found, return None
        return None

    # You will implement the following three methods in the exercises below.
    def add_request(self, prompt: str) -> str:
        raise NotImplementedError("This method should be implemented in Exercise 1")

    def process_requests(self, max_requests: int) -> None:
        raise NotImplementedError("This method should be implemented in Exercise 2")

    def get_completed_output(self, request_id: str) -> Optional[str]:
        raise NotImplementedError("This method should be implemented in Exercise 3")

### Exercise 1: `add_request`

Your first task is to implement the `add_request` method. This is the main entry point for users submitting new prompts to our engine. This method should:
1.  Generate a new, unique `request_id`.
2.  Create a `Request` object with the given prompt, the new ID, and an initial status of `'pending'`.
3.  Add the new `Request` object to the `_request_queue`.
4.  Return the new `request_id` to the user.

**Hint:** Use `next(self._request_id_counter)` to get a unique integer, and remember to convert it to a string to use as the `request_id`. New items should be added to the right side of a `deque`.

In [ ]:
def add_request_implementation(self, prompt: str) -> str:
    """
    Adds a new prompt to the request queue.

    Args:
        prompt: The user's prompt string.

    Returns:
        A unique request_id for the newly created request.
    """


# Monkey-patch the implementation into our class
RequestManager.add_request = add_request_implementation

In [ ]:
# TEST CELL FOR EXERCISE 1

manager = RequestManager()

# Add a first request
prompt1 = "What is the capital of France?"
id1 = manager.add_request(prompt1)
assert id1 == "0", f"Expected first request ID to be '0', but got '{id1}'"
print("✅ Test 1 passed: First request ID is correct.")

# Add a second request
prompt2 = "Who wrote the play Hamlet?"
id2 = manager.add_request(prompt2)
assert id2 == "1", f"Expected second request ID to be '1', but got '{id2}'"
print("✅ Test 2 passed: Second request ID is correct.")

# Check the internal state
assert len(manager._request_queue) == 2, f"Expected queue length to be 2, but got {len(manager._request_queue)}"
print("✅ Test 3 passed: Queue contains the correct number of requests.")

# Check the status of the newly added requests using the helper method
status1 = manager.get_request_status(id1)
assert status1 == "pending", f"Expected request '{id1}' to have status 'pending', but got '{status1}'"
status2 = manager.get_request_status(id2)
assert status2 == "pending", f"Expected request '{id2}' to have status 'pending', but got '{status2}'"
print("✅ Test 4 passed: New requests have the 'pending' status.")

print("\nAll tests for Exercise 1 passed! Great job!")

### Exercise 2: `process_requests`

Excellent! Now that you can queue requests, you need a way to process them. This method simulates the core vLLM engine taking a batch of requests from the queue, running inference, and marking them as complete.

Your `process_requests` method should:
1.  Determine how many requests to process. This should be the smaller of `max_requests` or the current number of requests in the queue.
2.  Loop that many times. In each iteration:
    a. Remove one `Request` from the *front* of the `_request_queue`.
    b. Change its status to `'completed'`.
    c. Store it in the `_completed_requests` dictionary, using its `request_id` as the key.

**Hint:** A `deque` has a `popleft()` method that is perfect for removing items from the front (left side) of the queue.

In [ ]:
def process_requests_implementation(self, max_requests: int) -> None:
    """
    Processes a batch of requests from the front of the queue.
    
    Args:
        max_requests: The maximum number of requests to process in this batch.
    """
        # 1. Get the next request from the front of the queue.
        request_to_process = self._request_queue.popleft()

        # 2. Update its status to 'completed'.
        request_to_process.status = 'completed'

        # 3. Move it to the completed requests dictionary.
        # The key should be the request_id.
        self._completed_requests[request_to_process.request_id] = request_to_process

# Monkey-patch the implementation into our class
RequestManager.process_requests = process_requests_implementation

In [ ]:
# TEST CELL FOR EXERCISE 2

manager = RequestManager()
# Add 5 requests for the test
for i in range(5):
    manager.add_request(f"Prompt {i}")

# Process a batch of 3 requests
manager.process_requests(max_requests=3)

# Check the state of the queues
assert len(manager._request_queue) == 2, f"Expected 2 requests left in the queue, but found {len(manager._request_queue)}"
print("✅ Test 1 passed: Correct number of requests remain in the queue.")

assert len(manager._completed_requests) == 3, f"Expected 3 requests in the completed dict, but found {len(manager._completed_requests)}"
print("✅ Test 2 passed: Correct number of requests moved to completed.")

# Check the status of individual requests
assert manager.get_request_status("0") == "completed", "Request '0' should be completed."
assert manager.get_request_status("1") == "completed", "Request '1' should be completed."
assert manager.get_request_status("2") == "completed", "Request '2' should be completed."
print("✅ Test 3 passed: The first 3 requests are marked as 'completed'.")

assert manager.get_request_status("3") == "pending", "Request '3' should still be pending."
assert manager.get_request_status("4") == "pending", "Request '4' should still be pending."
print("✅ Test 4 passed: The last 2 requests are still 'pending'.")

# Test processing remaining requests
manager.process_requests(max_requests=5) # Request more than available
assert len(manager._request_queue) == 0, "Queue should be empty after processing all requests."
assert len(manager._completed_requests) == 5, "All 5 requests should now be in the completed dict."
assert manager.get_request_status("4") == "completed", "Request '4' should now be completed."
print("✅ Test 5 passed: Processing remaining requests works correctly.")

print("\nAll tests for Exercise 2 passed! You're doing great!")

### Exercise 3: `get_completed_output`

You're on the home stretch! The final step is to create a way for users to retrieve the results of their requests.

Implement the `get_completed_output` method. It should:
1.  Check if the given `request_id` exists in the `_completed_requests` dictionary.
2.  If it exists, return a formatted string that simulates the LLM's output (e.g., `"Output for prompt: '...'"`).
3.  If the `request_id` is not in the completed dictionary (meaning it's either pending or invalid), return `None`.

In [ ]:
def get_completed_output_implementation(self, request_id: str) -> Optional[str]:
    """
    Retrieves the output for a completed request.

    Args:
        request_id: The ID of the request to retrieve.

    Returns:
        A string containing the mocked output if the request is complete,
        otherwise None.
    """


# Monkey-patch the implementation into our class
RequestManager.